# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rifkiay/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page, on one specific day, for one client.**
This is the grain of `fact_content_daily_performance`: `report_date × client_hash_id × content_hash_id`.

**Time window:** developing on a mid-panel month, `month=2026-03`, to avoid the sealed final month (`2026-06`, which is what the `_sample` table actually is).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

query_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""
result = con.sql(query_grain).df()
print("Duplicate grain rows found:", len(result))
result

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,n


Verified: 0 duplicate rows found for the grain `report_date × client_hash_id × content_hash_id` in the `month=2026-03` slice — confirming one row really does mean one page, one client, one day.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (available before the decision point):**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`, `scroll_events`, `sessions_ai` — daily observed signals from prior days

**Label / proxy:**
- Decline status, built from comparing `gsc_impressions` across time windows

**Context:**
- `client_hash_id`, `content_hash_id` — joins and grouped validation only
- `gsc_data_available`, `ga4_data_available` — used to filter/understand coverage, checked with `IS TRUE`

**Excluded (and why):**
- Any FlyRank product output (`health_score`, `priority_score`, `action_type`) — not shipped in this data, excluded on principle.
- `report_date` as a raw feature — could let the model memorize calendar periods instead of learning a pattern.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries verify the contract above: grain (done in Section 1), row count + date span, and availability using `IS TRUE`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query_count_dates = f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(query_count_dates).df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [4]:
query_availability = f"""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(query_availability).df()

,total_rows,ga4_available_rows
0,9841378,413966.0


In [5]:
features_df = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,                                              -- knowable because: past-day observed count, already logged
    gsc_avg_position,                                             -- knowable because: search position from a completed day
    CASE WHEN gsc_impressions > 0 
         THEN gsc_clicks * 1.0 / gsc_impressions ELSE 0 END AS ctr,          -- knowable because: derived from already-recorded clicks/impressions
    CASE WHEN ga4_sessions > 0 
         THEN ga4_engaged_sessions * 1.0 / ga4_sessions ELSE 0 END AS engagement_rate,  -- knowable because: derived from completed GA4 sessions
    CASE WHEN ga4_pageviews > 0 
         THEN scroll_events * 1.0 / ga4_pageviews ELSE 0 END AS scroll_rate  -- knowable because: derived from completed scroll events
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
LIMIT 1000
""").df()

features_df.head()

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate
0,content_09be8cc7fcb222af,client_65de48885f4ef01b,2026-03-01,0,NaN,0.0,0.0,0.0
1,content_851afac9fe13612e,client_65de48885f4ef01b,2026-03-01,0,NaN,0.0,0.0,0.0
2,content_cee6c6fc8c51af14,client_65de48885f4ef01b,2026-03-01,0,NaN,0.0,0.0,0.0
3,content_5e120e972f11f833,client_65de48885f4ef01b,2026-03-01,0,NaN,0.0,0.0,0.0
4,content_16a7291bb6ecaebe,client_65de48885f4ef01b,2026-03-01,0,NaN,0.0,0.0,0.0


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features_df["is_declining"] = (features_df["gsc_impressions"] < features_df["gsc_impressions"].median()).astype(int)

X_honest = features_df[["gsc_avg_position", "ctr", "engagement_rate", "scroll_rate"]].fillna(0)
y = features_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model = LogisticRegression().fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest ROC AUC:", honest_score)

X_leaky = features_df[["gsc_avg_position", "ctr", "engagement_rate", "scroll_rate", "gsc_impressions"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = LogisticRegression().fit(X_train, y_train)
leaky_score = roc_auc_score(y_test, model_leaky.predict_proba(X_test)[:, 1])
print("Leaky ROC AUC (with gsc_impressions as a feature):", leaky_score)

print(f"\nScore jumped from {honest_score:.3f} to {leaky_score:.3f} once I added a column")
print("that was literally used to build the label. Removing it and keeping the honest number.")

Honest ROC AUC: 0.5816306089743589
Leaky ROC AUC (with gsc_impressions as a feature): 1.0

Score jumped from 0.582 to 1.000 once I added a column
that was literally used to build the label. Removing it and keeping the honest number.


The leak: `gsc_impressions` was used to build `is_declining` directly, so including it as a feature let the model "cheat" — ROC AUC jumped from 0.582 to a perfect 1.000. That 1.000 score isn't a real model skill, it's the model literally seeing the answer. Removing that column and keeping the honest 0.582 as the real result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** this slice covers only one month (`2026-03`), so it can't capture seasonality — a page's real growth/decline tendency could look different in another month. Also, `ga4_data_available IS TRUE` only covers 413,966 of 9,841,378 rows (~4.2%) — most of this month's data is GSC-only, so any feature built from GA4 columns (engagement_rate, scroll_rate) applies to a small slice, not the whole panel.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.